# DM-KG: Dvoretzky-Milman Projections for Knowledge Graphs

## Analysis Notebook

This notebook analyzes the performance of the **DM-KG model** against two baselines:
- **Euclidean TransE** — fast but poor at hierarchical relations
- **Hyperbolic TransE** — accurate but slow (Möbius operations)

The key research question: *Can DM projections give near-hyperbolic accuracy at near-Euclidean speed?*

---

### Contents
1. Benchmark Results: Speed Comparison
2. Accuracy Projection: Expected MRR vs Measured MRR
3. Speed-vs-Accuracy Tradeoff
4. Projection Dimension Ablation
5. Per-Relation Analysis

In [ ]:
# Setup
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), '..') if 'notebooks' in os.getcwd() else '.')
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Try seaborn for nicer plots
try:
    import seaborn as sns
    sns.set_style('whitegrid')
    sns.set_context('notebook', font_scale=1.2)
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['savefig.dpi'] = 150
matplotlib.rcParams['figure.figsize'] = (10, 6)

print(f'PyTorch: {torch.__version__}')
print(f'Seaborn: {"available" if HAS_SEABORN else "not available"}')


## 1. Benchmark Results: Speed Comparison

Load the benchmark results generated by `scripts/benchmark.py` and display a comparison table.

In [ ]:
# Try to load benchmark results; use synthetic data if not available
benchmark_path = '../results/benchmark.json'
try:
    with open(benchmark_path) as f:
        results = json.load(f)
    df = pd.DataFrame(results)
    print('Loaded benchmark results:')
except FileNotFoundError:
    print('No benchmark.json found. Using illustrative data (run scripts/benchmark.py first).')
    # Illustrative data based on expected relative performance
    results = [
        {'model': 'Euclidean TransE', 'num_params': 3800000, 'forward_mean_ms': 1.2, 'epoch_time_sec': 45.0, 'samples_per_sec': 50000, 'peak_memory_mb': 450},
        {'model': 'DM-KG',            'num_params': 3900000, 'forward_mean_ms': 2.8, 'epoch_time_sec': 95.0, 'samples_per_sec': 24000, 'peak_memory_mb': 520},
        {'model': 'Hyperbolic TransE','num_params': 3750000, 'forward_mean_ms': 18.5,'epoch_time_sec': 580.0,'samples_per_sec': 3900, 'peak_memory_mb': 680},
    ]
    df = pd.DataFrame(results)
    print('Using illustrative data:')
df.set_index('model', inplace=True)
display(df)


In [ ]:
# Bar chart: Forward pass time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = df.index.tolist()
colors = ['#2ecc71', '#3498db', '#e74c3c']

# Forward time
ax = axes[0]
bars = ax.bar(models, df['forward_mean_ms'], color=colors, edgecolor='white', linewidth=1.2)
ax.set_ylabel('Forward Pass Time (ms)', fontweight='bold')
ax.set_title('Forward Pass Time Comparison', fontweight='bold', fontsize=14)
for bar, val in zip(bars, df['forward_mean_ms']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.1f} ms',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

# Epoch time
ax = axes[1]
bars = ax.bar(models, df['epoch_time_sec'], color=colors, edgecolor='white', linewidth=1.2)
ax.set_ylabel('Epoch Time (seconds)', fontweight='bold')
ax.set_title('Training Epoch Time Comparison', fontweight='bold', fontsize=14)
for bar, val in zip(bars, df['epoch_time_sec']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val:.0f}s',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

# Speedup calculation
dm = df.loc['DM-KG', 'forward_mean_ms']
hyp = df.loc['Hyperbolic TransE', 'forward_mean_ms']
euc = df.loc['Euclidean TransE', 'forward_mean_ms']
print(f'DM-KG is {hyp/dm:.1f}x faster than Hyperbolic TransE (forward pass)')
print(f'DM-KG is {dm/euc:.1f}x slower than Euclidean TransE (forward pass)')


## 2. Speed-vs-Accuracy Tradeoff

The key insight: DM projections should sit between Euclidean (fast, low accuracy) and Hyperbolic (slow, high accuracy) — giving us the **best of both worlds**.

> Data below is illustrative. Replace with actual MRR values from training runs.

In [ ]:
# Speed-vs-Accuracy scatter plot
# Expected MRR on FB15k-237 (illustrative — replace with your actual results)
accuracy_data = {
    'Euclidean TransE':  {'mrr': 0.279, 'fwd_ms': 1.2},
    'DM-KG':             {'mrr': 0.312, 'fwd_ms': 2.8},
    'Hyperbolic TransE': {'mrr': 0.321, 'fwd_ms': 18.5},
}

fig, ax = plt.subplots(figsize=(10, 7))

for i, (name, d) in enumerate(accuracy_data.items()):
    ax.scatter(d['fwd_ms'], d['mrr'], s=400, c=colors[i], edgecolors='black',
               linewidth=2, zorder=5, label=name)
    offset_x = 0.8 if name != 'Hyperbolic TransE' else -3.5
    offset_y = 0.003
    ax.annotate(name, (d['fwd_ms'], d['mrr']),
                (d['fwd_ms'] + offset_x, d['mrr'] + offset_y),
                fontsize=13, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Draw the Pareto frontier idea
pareto_x = [accuracy_data['Euclidean TransE']['fwd_ms'],
            accuracy_data['DM-KG']['fwd_ms'],
            accuracy_data['Hyperbolic TransE']['fwd_ms']]
pareto_y = [accuracy_data['Euclidean TransE']['mrr'],
            accuracy_data['DM-KG']['mrr'],
            accuracy_data['Hyperbolic TransE']['mrr']]
ax.plot(pareto_x, pareto_y, 'k--', alpha=0.3, linewidth=2, label='Pareto frontier')

ax.set_xlabel('Forward Pass Time (ms, lower is better)', fontsize=13, fontweight='bold')
ax.set_ylabel('MRR (higher is better)', fontsize=13, fontweight='bold')
ax.set_title('Speed vs. Accuracy Tradeoff\n(DM-KG aims for the sweet spot)', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')

# Add shaded region for "sweet spot"
ax.axvspan(1.5, 5, alpha=0.08, color='green', label='Sweet spot')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Key takeaway: DM-KG should achieve ~97% of hyperbolic MRR at ~15% of the forward-pass cost.')


## 3. Projection Dimension Ablation

How does the projected dimension $k$ affect accuracy? The Dvoretzky-Milman theorem predicts that accuracy should converge to the full hyperbolic accuracy rapidly as $k$ increases.

We sweep $k \in \{8, 16, 32, 64, 128\}$ with a fixed hyperbolic dimension $d = 256$.

In [ ]:
# Projection dimension ablation (illustrative — run actual experiments for real data)
k_values = [8, 16, 32, 64, 128]

# Illustrative data showing convergence pattern
# In real experiments, replace with actual MRR from training runs
mrr_values = [0.285, 0.302, 0.312, 0.317, 0.319]  # DM-KG at different k
hyperbolic_mrr = 0.321  # Full hyperbolic baseline MRR (k = d = 256 effectively)
euclidean_mrr = 0.279   # Euclidean baseline

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(k_values, mrr_values, 'o-', color='#3498db', linewidth=2.5, markersize=10,
        label='DM-KG (varying $k$)', zorder=5)
ax.axhline(y=hyperbolic_mrr, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Hyperbolic TransE (MRR={hyperbolic_mrr:.3f})')
ax.axhline(y=euclidean_mrr, color='#2ecc71', linestyle='--', linewidth=2,
           label=f'Euclidean TransE (MRR={euclidean_mrr:.3f})')

ax.set_xlabel('Projection Dimension $k$', fontsize=13, fontweight='bold')
ax.set_ylabel('MRR', fontsize=13, fontweight='bold')
ax.set_title('MRR vs. Projection Dimension $k$\n(Convergence to hyperbolic accuracy)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xscale('log', base=2)
ax.set_xticks(k_values)
ax.set_xticklabels([str(k) for k in k_values])
ax.grid(True, alpha=0.3)

# Annotate the point of diminishing returns
ax.annotate('Diminishing returns\nafter k=32',
            xy=(32, 0.312), xytext=(16, 0.295),
            fontsize=11, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

plt.tight_layout()
plt.show()

# Compute relative MRR
print('Projection Dimension Analysis:')
for k, mrr in zip(k_values, mrr_values):
    pct_of_hyperbolic = mrr / hyperbolic_mrr * 100
    pct_of_euclidean = (mrr - euclidean_mrr) / (hyperbolic_mrr - euclidean_mrr) * 100
    print(f'  k={k:3d}: MRR={mrr:.4f} ({pct_of_hyperbolic:.1f}% of hyperbolic, captures {pct_of_euclidean:.0f}% of the gap)')


## 4. Memory Usage Comparison

GPU memory is a critical resource. Hyperbolic models need extra memory for Möbius operations and intermediate tensors.

In [ ]:
# Memory usage bar chart
if 'peak_memory_mb' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(df.index, df['peak_memory_mb'], color=colors, edgecolor='white', linewidth=1.2)
    ax.set_ylabel('Peak GPU Memory (MB)', fontweight='bold')
    ax.set_title('GPU Memory Usage Comparison', fontweight='bold', fontsize=14)
    for bar, val in zip(bars, df['peak_memory_mb']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val:.0f} MB',
                ha='center', va='bottom', fontweight='bold', fontsize=11)

    # Efficiency: MRR per MB
    # Using illustrative MRR values
    mrr_vals = {'Euclidean TransE': 0.279, 'DM-KG': 0.312, 'Hyperbolic TransE': 0.321}
    efficiency = {m: mrr_vals[m] / (df.loc[m, 'peak_memory_mb']/100) for m in df.index}
    print('\nMRR Efficiency (MRR per 100MB):')
    for m, eff in efficiency.items():
        print(f'  {m}: {eff:.4f}')

    plt.tight_layout()
    plt.show()
else:
    print('Memory data not available (benchmark on GPU to collect).')


## 5. Summary & Next Steps

### Key Findings (Expected)
1. **DM-KG achieves ~97% of hyperbolic MRR** while being **~6.6× faster** on the forward pass
2. The **sweet spot** for projection dimension is around $k = 32$ for $d = 256$ hyperbolic space
3. Beyond $k = 64$, diminishing returns set in — accuracy gains are marginal vs. computational cost

### Next Steps
- Train on full FB15k-237 for 200 epochs to get real MRR numbers
- Add WN18RR dataset for hierarchical benchmark
- Ablate: fixed vs. learned projection matrices
- Analyze per-relation hyperbolicity ($\delta$) vs. optimal projection dimension
- Extend to full GNN message-passing (beyond TransE scoring)

### Citation
> Vershynin, R. (2018). *High-Dimensional Probability: An Introduction with Applications in Data Science*. Cambridge University Press.